# Finding Locations

In [1]:
# connect to Enterprise GIS
from arcgis.gis import GIS
import arcgis.geoanalytics

portal_gis = GIS("https://ndhwks6.esri.com/portal", "admin", 'esri.agp', verify_cert=False)

In [2]:
bigdata_datastore_manager = arcgis.geoanalytics.get_datastores()
bigdata_datastore_manager

<DatastoreManager for https://ndhwks6.esri.com:6443/arcgis/admin>

In [3]:
data_item2 = bigdata_datastore_manager.add_bigdata("all_hurricanes", r"\\NDHWKS6\Users\arcgis\Documents\hurricanes_1848_2010")

Big Data file share exists for all_hurricanes


In [4]:
search_result = portal_gis.content.search("bigDataFileShares_all_hurricanes", 
                                          item_type = "big data file share", 
                                          max_items=40)
search_result

[<Item title:"bigDataFileShares_all_hurricanes" type:Big Data File Share owner:admin>]

In [5]:
data_item = search_result[0]
data_item

<Item title:"bigDataFileShares_all_hurricanes" type:Big Data File Share owner:admin>

In [6]:
#displays layers in the item
data_item.layers

[<Layer url:"https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_all_hurricanes/BigDataCatalogServer/hurricanes">]

In [7]:
hurricanes = data_item.layers[0] #select first layer 
hurricanes

<Layer url:"https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_all_hurricanes/BigDataCatalogServer/hurricanes">

In [8]:
search_result = portal_gis.content.get('8f2fd1d2488f47adbe07a3ddcb05e24e')

In [9]:
table = search_result.tables[0]
table

<Table url:"https://ndhwks6.esri.com/server/rest/services/Hosted/ImportantPlaces/FeatureServer/1">

## Detect Incidents

In [10]:
from arcgis.geoanalytics.find_locations import detect_incidents

In [11]:
##usage example
incidents_detected = detect_incidents(input_layer=hurricanes, 
                                      track_fields='track_type',
                                      start_condition_expression='$feature["Wind"] < 0.2')

In [13]:
incidents_detected.delete()

True

## Geocode Locations

In [14]:
from arcgis.geoanalytics.find_locations import geocode_locations

In [71]:
# ?bx.geocode_locations(input_layer=table, country='CA', output_name='geocoded')

TypeError: geocode_locations() got an unexpected keyword argument 'country'

## Find Dwell Locations

In [16]:
from arcgis.geoanalytics.find_locations import find_dwell_locations
from datetime import datetime as dt

In [17]:
dwell_locs = find_dwell_locations(input_layer=hurricanes,
                                  track_fields='track_type',
                                  distance_tolerance=1,
                                  distance_unit='Meters',
                                  time_tolerance='1',
                                  time_unit='Hours', 
                                  output_name='dwell locations' + str(dt.now().microsecond))

In [18]:
dwell_locs.delete()

True

## Find Similar Locations

In [19]:
from arcgis.geoanalytics.find_locations import find_similar_locations 

In [20]:
data_item2 = bigdata_datastore_manager.add_bigdata("Chicago_Crimes", r"\\NDHWKS6\Users\arcgis\Documents\ga-store1")

Big Data file share exists for Chicago_Crimes


In [21]:
search_result = portal_gis.content.search("bigDataFileShares_Chicago_Crimes", 
                                          item_type = "big data file share", 
                                          max_items=40)
search_result

[<Item title:"bigDataFileShares_Chicago_Crimes_2" type:Big Data File Share owner:admin>,
 <Item title:"bigDataFileShares_Chicago_Crimes" type:Big Data File Share owner:admin>]

In [22]:
homicides = portal_gis.content.get('79a9c31548cb4de2a17b06a9e67095ba')
homicides

<Item title:"Homicides" type:Feature Layer Collection owner:admin>

In [23]:
layers = search_result[0].layers
layers

[<Layer url:"https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_Chicago_Crimes_2/BigDataCatalogServer/crime">]

In [24]:
crime_incidents = layers[0]
homicides = homicides.layers[0]

In [25]:
similar_locs = find_similar_locations(input_layer=crime_incidents,
                                      search_layer=homicides,
                                      analysis_fields='Beat',
                                      most_or_least_similar='MostSimilar',
                                      match_method='AttributeValues',
                                      number_of_results=10,
                                      return_tuple=True,
                                      output_name='similar locations'+ str(dt.now().microsecond))